# Giving Agents Tools and Knowledge: MCP, Skills and OKF

The runnable code of [Giving Agents Tools and Knowledge: MCP, Skills and OKF](https://residual-stream.lvtan71.workers.dev/posts/mcp-skills-and-okf/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/mcp-skills-and-okf.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.
The post's run read the blog's working tree with drafts in it; this snapshot is the last commit before, so the search scores differ in the last digits.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q mcp==2.3.0

# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "mcp-skills-and-okf"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/mcp-skills-and-okf/blog_server.py', 'data/blog-deb8375.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## An MCP server for this blog

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/mcp-skills-and-okf/#an-mcp-server-for-this-blog)

In [ ]:
%%writefile blog_server.py
import glob
import math
import os
import re
from collections import Counter
from mcp.server.mcpserver import MCPServer

# An MCP server for this blog: two tools and one resource, over stdio. Any
# MCP host (an IDE, a desktop assistant, an agent) can launch it and use
# them, without code written for that host.
POSTS = os.environ.get("POSTS", "content/posts")

def load():
    out = {}
    for path in glob.glob(f"{POSTS}/*.mdx"):
        raw = open(path).read()
        title = re.search(r'^title: "(.*)"$', raw, re.M)
        body = re.sub(r"```.*?```|\$\$.*?\$\$|^import .*?$", " ", raw.split("---", 2)[-1], flags=re.S | re.M)
        out[os.path.basename(path)[:-4]] = (title.group(1) if title else "", " ".join(body.split()))
    return out
posts = load()
words = lambda t: re.findall(r"\w+", t.lower())
counts = {s: Counter(words(b)) for s, (_, b) in posts.items()}
df = Counter(w for c in counts.values() for w in c)
avg = sum(sum(c.values()) for c in counts.values()) / len(counts)

server = MCPServer("residual-stream-blog", instructions="Search and read the Residual Stream blog's posts.")

@server.tool()
def search_posts(query: str, k: int = 3) -> list[dict]:
    """Find the blog posts most relevant to a query (BM25 over the full text).
    Returns each post's slug, title and score."""
    def bm25(slug):
        c, n = counts[slug], sum(counts[slug].values())
        return sum(math.log((len(posts) - df[w] + 0.5) / (df[w] + 0.5) + 1) * c[w] * 2.5
                   / (c[w] + 1.5 * (0.25 + 0.75 * n / avg)) for w in set(words(query)) if w in df)
    best = sorted(posts, key=bm25, reverse=True)[:k]
    return [{"slug": s, "title": posts[s][0], "score": round(bm25(s), 2)} for s in best]

@server.tool()
def read_post(slug: str, max_words: int = 400) -> str:
    """Read the start of one post, by slug."""
    if slug not in posts:
        raise ValueError(f"no post {slug!r}")
    return " ".join(posts[slug][1].split()[:max_words])

@server.resource("post://{slug}")
def post_resource(slug: str) -> str:
    """A post's full text, for a host to attach as context."""
    return posts[slug][1]

if __name__ == "__main__":
    server.run()                                  # stdio by default

In [ ]:
# blog_server.py is a stdio server, not run here: the next cells launch it as a
# subprocess, and later ones import it.

In [ ]:
import asyncio
import json
import sys
from mcp import Client, StdioServerParameters

# A host's side: launch the server as a subprocess, discover what it offers,
# and call a tool, all over JSON-RPC on the subprocess's stdin and stdout.
async def main():
    params = StdioServerParameters(command=sys.executable, args=["blog_server.py"])
    async with Client(params) as client:
        tools = await client.list_tools()
        for t in tools.tools:
            print(f"tool {t.name}: {t.description.splitlines()[0]}")
            print(f"  input schema: {json.dumps(t.input_schema)[:160]}...")
        print(f"protocol version: {client.protocol_version}")
        result = await client.call_tool("search_posts", {"query": "paged attention KV cache memory", "k": 3})
        print("search_posts ->", result.structured_content or result.content[0].text)
        text = await client.call_tool("read_post", {"slug": "kv-cache-compression", "max_words": 25})
        print("read_post ->", text.content[0].text)
        # The resource: the host, not the model, decides to read it and attach it.
        for t in (await client.list_resource_templates()).resource_templates:
            print(f"resource template {t.uri_template}: {t.description}")
        res = await client.read_resource("post://mixture-of-experts")
        print("resources/read ->", res.contents[0].text[:80], "...")
await main()  # notebook: Jupyter already runs an event loop

The post printed:

```text
tool search_posts: Find the blog posts most relevant to a query (BM25 over the full text).
  input schema: {"type": "object", "properties": {"query": {"title": "Query", "type": "string"}, "k": {"default": 3, "title": "K", "type": "integer"}}, "required": ["query"], "...
tool read_post: Read the start of one post, by slug.
  input schema: {"type": "object", "properties": {"slug": {"title": "Slug", "type": "string"}, "max_words": {"default": 400, "title": "Max Words", "type": "integer"}}, "require...
protocol version: 2026-07-28
search_posts -> {'result': [{'slug': 'paged-attention-explained', 'title': 'Paged Attention: Managing the KV Cache Like Virtual Memory', 'score': 5.75}, {'slug': 'kv-cache-explained', 'title': 'The KV Cache: How Transformers Avoid Recomputing the Past', 'score': 5.44}, {'slug': 'flashattention-explained', 'title': 'FlashAttention: Exact Attention at the Speed of Memory', 'score': 4.88}]}
read_post -> [Post 20](/posts/kv-cache-explained/) sized the KV cache, and posts 18 and 23 showed the architectural ways to shrink it: share key-value heads (GQA), or store a
resource template post://{slug}: A post's full text, for a host to attach as context.
resources/read -> [Post 13](/posts/residual-stream-explained/) found two thirds of a Transformer b ...
```

In [ ]:
import json
import subprocess
import sys

# The same exchange without an SDK: JSON-RPC 2.0 messages, one per line, on
# the server's stdin and stdout. In the 2026-07-28 protocol every request is
# self-contained: its _meta carries the protocol version and the client's
# capabilities, so no handshake has to come first.
proc = subprocess.Popen([sys.executable, "blog_server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                        stderr=subprocess.DEVNULL, text=True)
meta = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "wire-demo", "version": "0.1"}}
request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
           "params": {"name": "search_posts", "arguments": {"query": "speculative decoding draft", "k": 2}, "_meta": meta}}
print("->", json.dumps(request))
proc.stdin.write(json.dumps(request) + "\n"); proc.stdin.flush()
reply = json.loads(proc.stdout.readline())
print("<-", json.dumps(reply)[:700])
proc.terminate()

The post printed:

```text
-> {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search_posts", "arguments": {"query": "speculative decoding draft", "k": 2}, "_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}, "io.modelcontextprotocol/clientInfo": {"name": "wire-demo", "version": "0.1"}}}}
<- {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"text": "{\n  \"slug\": \"speculative-decoding\",\n  \"title\": \"Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE\",\n  \"score\": 10.61\n}", "type": "text"}, {"text": "{\n  \"slug\": \"tensorrt-llm-explained\",\n  \"title\": \"TensorRT-LLM: Compiling Inference Engines Ahead of Time\",\n  \"score\": 10.05\n}", "type": "text"}], "isError": false, "resultType": "complete", "structuredContent": {"result": [{"slug": "speculative-decoding", "title": "Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE", "score": 10.61}, {"slug": "tensorrt-llm-explained", "title": "TensorRT-LLM: Compiling Inference Engine
```

## What tools cost in context

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/mcp-skills-and-okf/#what-tools-cost-in-context)

In [ ]:
import asyncio
import json
from transformers import AutoTokenizer
from mcp import Client
import blog_server
from huggingface_hub import snapshot_download

# What tool and skill definitions cost in context, in tokens (Qwen2.5's
# tokenizer, as a stand-in for any model's). Every tool a host connects is
# described in the prompt on every call, used or not.
tok = AutoTokenizer.from_pretrained(snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                                                      revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"))
n = lambda text: len(tok(text).input_ids)

async def tool_defs():
    async with Client(blog_server.server) as client:                 # in-process, same server as before
        return [t.model_dump(by_alias=True, exclude_none=True) for t in (await client.list_tools()).tools]
defs = await tool_defs()  # notebook: Jupyter already runs an event loop
per_tool = sum(n(json.dumps(d)) for d in defs) / len(defs)
print(f"this blog's 2 tools: {sum(n(json.dumps(d)) for d in defs)} tokens of definitions, {per_tool:.0f} per tool")
no_out = sum(n(json.dumps({k: v for k, v in d.items() if k != "outputSchema"})) for d in defs)
print(f"  of which output schemas: {sum(n(json.dumps(d)) for d in defs) - no_out} tokens")
print(f"5 servers x 15 tools at that size: {5 * 15 * per_tool:,.0f} tokens in every request's prompt")

meta = ("name: pdf-processing\ndescription: Extract text and tables from PDF files, fill forms, merge documents. "
        "Use when working with PDF files or when the user mentions PDFs, forms, or document extraction.")
print(f"\none skill's metadata (name and description): {n(meta)} tokens")
for skills in (10, 50, 200):
    print(f"{skills:4d} skills: metadata always loaded {skills * n(meta):7,d} tokens; "
          f"every body loaded up front at ~4,000 tokens each {skills * 4000:9,d} tokens")

The post printed:

```text
this blog's 2 tools: 295 tokens of definitions, 148 per tool
  of which output schemas: 101 tokens
5 servers x 15 tools at that size: 11,062 tokens in every request's prompt

one skill's metadata (name and description): 41 tokens
  10 skills: metadata always loaded     410 tokens; every body loaded up front at ~4,000 tokens each    40,000 tokens
  50 skills: metadata always loaded   2,050 tokens; every body loaded up front at ~4,000 tokens each   200,000 tokens
 200 skills: metadata always loaded   8,200 tokens; every body loaded up front at ~4,000 tokens each   800,000 tokens
```

## OKF: curated knowledge, as files

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/mcp-skills-and-okf/#okf-curated-knowledge-as-files)

In [ ]:
import json
import os
import yaml

# An OKF bundle (Open Knowledge Format v0.2) for a few of this blog's
# concepts: a directory of Markdown files, each with YAML frontmatter whose
# only required field is `type`, plus an index. Generated from the posts'
# own front matter, so it records where it came from and that no person has
# checked it yet. The concepts are ideas, not assets, so they carry no
# `resource`; the post each was derived from goes in `sources`.
POSTS = "content/posts"
SITE = "https://residual-stream.lvtan71.workers.dev"
concepts = {"kv-cache": "kv-cache-explained", "lora": "parameter-efficient-fine-tuning-lora",
            "speculative-decoding": "speculative-decoding", "hnsw": "vector-search-and-ann-indexes"}
os.makedirs("bundle/concepts", exist_ok=True)
index = ["---", "okf_version: \"0.2\"", "---", "", "# Residual Stream concepts", ""]
for name, slug in concepts.items():
    path = f"{POSTS}/{slug}.mdx"
    if not os.path.exists(path):
        continue                                              # a post not yet written (lora, here)
    front = yaml.safe_load(open(path).read().split("---")[1])
    title, desc = front["title"], " ".join(front["description"].split())
    doc = f"""---
type: Concept
title: {json.dumps(title, ensure_ascii=False)}
description: {json.dumps(desc[:200] + "...", ensure_ascii=False)}
tags: [llm, inference]
sources:
  - resource: {SITE}/posts/{slug}/
    title: {json.dumps(title, ensure_ascii=False)}
generated:
  by: process:make_okf
  at: "2026-10-01"
status: draft
---

# Summary

{desc}

# See also

[the full post]({SITE}/posts/{slug}/)
"""
    open(f"bundle/concepts/{name}.md", "w").write(doc)
    index.append(f"- [{title}](concepts/{name}.md)")
open("bundle/index.md", "w").write("\n".join(index) + "\n")
print(open("bundle/index.md").read())
print(open("bundle/concepts/kv-cache.md").read()[:900])

The post printed:

```text
---
okf_version: "0.2"
---

# Residual Stream concepts

- [The KV Cache: How Transformers Avoid Recomputing the Past](concepts/kv-cache.md)
- [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](concepts/speculative-decoding.md)
- [Vector Search: Exact Search and the Approximate Indexes](concepts/hnsw.md)

---
type: Concept
title: "The KV Cache: How Transformers Avoid Recomputing the Past"
description: "Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to linear, worked through with colored matrices step by step — plus the memory cost formula and what it mea..."
tags: [llm, inference]
sources:
  - resource: https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/
    title: "The KV Cache: How Transformers Avoid Recomputing the Past"
generated:
  by: process:make_okf
  at: "2026-10-01"
status: draft
---

# Summary

Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to linear, worked through with colored matrices step by step — plus the memory cost formula and what it means for real models.

# See also

[the full post](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/)
```